In [2]:
import os
import pandas as pd 
import re

In [3]:
# Config
DATASET = 'Mouse-2023' 
RAW_DIR = os.path.join(DATASET, 'raw-data')
PROCESSED_DIR = os.path.join(DATASET, 'processed-data')

**Make Counts Matrix**

In [58]:
counts_path = os.path.join(RAW_DIR, "gene_counts_matrix.tsv")
counts_df = pd.read_csv(counts_path, sep="\t", index_col=0)
counts_df.shape
# counts are in genes x samples format for DESeq2, need to transpose
counts_df = counts_df.T
counts_df.shape
# now counts are samples x genes (12 samples x 37991 genes)
counts_df.head()
# Add index name (sample)
counts_df.columns.name = None
counts_df.index.name = "sample"
# Save processed counts to a new file 
counts_df = counts_df.reset_index()
processed_counts_path = os.path.join(PROCESSED_DIR, "counts.csv")
counts_df.to_csv(processed_counts_path, index=False)

**Make Metadata**

In [59]:
# Extract sample IDs from the index of the counts dataframe
counts_df = pd.read_csv(processed_counts_path, index_col='sample')
sample_ids = counts_df.index.tolist()

# Add treatment column
def sample_map(n):
  if 142 <= n <= 145: return "control"
  elif 146 <= n <= 149: return "CFA"
  elif 150 <= n <= 153: return "CFB"

metadata = []
for sid in sample_ids: 
  match = re.search(r"(\d+)$", sid)
  n = int(match.group(1))
  treatment = sample_map(n)
  metadata.append({
    "sample_ID": sid,
    "treatment": treatment})
  
metadata_df = pd.DataFrame(metadata)

# Add replicate column
metadata_df['replicate'] = metadata_df.groupby('treatment').cumcount() + 1

# Add intuitive sample name 
metadata_df['sample'] = metadata_df['treatment'] + "_rep" + metadata_df['replicate'].astype(str)

# Reorder columns and save
cols = ['sample', 'sample_ID', 'treatment', 'replicate']
metadata_df = metadata_df[cols]
metadata_path = os.path.join(PROCESSED_DIR, "metadata.csv")
metadata_df.to_csv(metadata_path, index=False)

# Replace sample names in counts file 
counts_df = pd.read_csv(processed_counts_path, index_col=None)
counts_df['sample'] = metadata_df['sample']
counts_df.to_csv(processed_counts_path, index=False)

**Make QC Stats**

In [39]:
# Config 
mapping_file = os.path.join(RAW_DIR, "featureCounts_raw.txt.summary")
counts_file = os.path.join(PROCESSED_DIR, "counts.csv")
qc_out_path = os.path.join(PROCESSED_DIR, "qc_metrics.csv")

reads_threshold = 5e6
mapping_threshold = 0.5
rRNA_threshold = 0.15


In [61]:
# Read the files
mapping_df = pd.read_csv(mapping_file, sep="\t", header=0, index_col=0)
counts_df = pd.read_csv(counts_file, index_col='sample')

# Explore data structure and ensure it is read correctly
mapping_df.head() # Rows are mapping stats and columns are samples
mapping_df.shape # There are 14 mapping stats and 12 samples
mapping_df.columns = counts_df.index # Clean up sample names 

# Extract mapping stats names to a list for future use 
mapping_stats=mapping_df.index.tolist()
print(mapping_stats)

['Assigned', 'Unassigned_Unmapped', 'Unassigned_Read_Type', 'Unassigned_Singleton', 'Unassigned_MappingQuality', 'Unassigned_Chimera', 'Unassigned_FragmentLength', 'Unassigned_Duplicate', 'Unassigned_MultiMapping', 'Unassigned_Secondary', 'Unassigned_NonSplit', 'Unassigned_NoFeatures', 'Unassigned_Overlapping_Length', 'Unassigned_Ambiguity']


In [ ]:
qc_df = pd.DataFrame(index=counts_df.index)

# Mapping stats
qc_df['total_reads'] = mapping_df.sum(axis=0)
# qc_df['library_size'] = counts_df.sum(axis=1)
qc_df['library_size'] = mapping_df.loc['Assigned'] # The above is equivalent
qc_df['mapping_rate'] = mapping_df.loc['Assigned'] / qc_df['total_reads']
qc_df['n_detected_genes'] = (counts_df > 0).sum(axis=1)
qc_df['multimapping_rate'] = mapping_df.loc['Unassigned_MultiMapping'] / qc_df['total_reads']

MAD_thresholds = {}
for stat in qc_df.columns:
  median = qc_df[stat].median()
  mad = (qc_df[stat] - median).abs().median() * 1.4826
  lower = median - (2 * mad)
  higher = median + (2 * mad)
  MAD_thresholds[stat] = (lower, higher)
thresholds_df = pd.DataFrame(MAD_thresholds, index=['mad_lower', 'mad_upper']).T

qc_df.head()

In [63]:
for sample in qc_df.index:
  # Absolute thresholds
  a_reasons = []
  if qc_df.loc[sample, 'library_size'] < reads_threshold: a_reasons.append('low_reads')
  if qc_df.loc[sample, 'mapping_rate'] < mapping_threshold: a_reasons.append('low_mapping_rate')
  if qc_df.loc[sample, 'multimapping_rate'] > rRNA_threshold: a_reasons.append('high_multimapping_rate')
  if a_reasons ==[]: 
    qc_df.loc[sample, 'absolute_thresholds'] = 'Pass'
  else: 
    qc_df.loc[sample, 'absolute_thresholds'] = ','.join(a_reasons)
  # Relative thresholds 
  r_reasons = []
  if qc_df.loc[sample, 'library_size'] < thresholds_df.loc['library_size', 'mad_lower']: r_reasons.append('low_reads')
  if qc_df.loc[sample, 'mapping_rate'] < thresholds_df.loc['mapping_rate', 'mad_lower']: r_reasons.append('low_mapping_rate')
  if qc_df.loc[sample, 'multimapping_rate'] > thresholds_df.loc['multimapping_rate', 'mad_upper']: r_reasons.append('high_multimapping_rate')
  if qc_df.loc[sample, 'n_detected_genes'] < thresholds_df.loc['n_detected_genes', 'mad_lower']: r_reasons.append('low_n_detected_genes')
  if r_reasons ==[]:
    qc_df.loc[sample, 'relative_thresholds'] = 'Pass'
  else:
    qc_df.loc[sample, 'relative_thresholds'] = ','.join(r_reasons)
  
qc_df.to_csv(qc_out_path)

In [64]:
qc_df

,total_reads,library_size,mapping_rate,n_detected_genes,multimapping_rate,absolute_thresholds,relative_thresholds
sample,,,,,,,
control_rep1,31962864,21449031,0.671061,19729,0.198599,high_multimapping_rate,Pass
control_rep2,31327744,20800152,0.663953,19765,0.207073,high_multimapping_rate,Pass
control_rep3,33394761,22784846,0.682288,19599,0.189995,high_multimapping_rate,Pass
control_rep4,33931308,23272322,0.685866,19987,0.187551,high_multimapping_rate,Pass
CFA_rep1,34099257,22382159,0.656383,19898,0.211304,high_multimapping_rate,low_mapping_rate
CFA_rep2,29351348,19989107,0.681029,19698,0.195099,high_multimapping_rate,Pass
CFA_rep3,29550447,19903587,0.673546,19692,0.200642,high_multimapping_rate,Pass
CFA_rep4,29631856,20083569,0.677770,19836,0.195494,high_multimapping_rate,Pass
CFB_rep1,31503275,21102913,0.669864,19628,0.206820,high_multimapping_rate,Pass
